# 🏥 Medical LLM Assistant — Tech Challenge Fase 3

**Autor:** Felipe Huszar  
**Repositório:** https://github.com/felipe-huszar/medical-llm-assistant  

Assistente médico com LLM (Mistral fine-tuned via LoRA), orquestrado por **LangGraph** e memória via **ChromaDB**.

---

## 🔧 Modos de Execução
| Modo | GPU | Descrição |
|------|-----|-----------|
| `USE_MOCK_LLM=true` | ❌ Não precisa | Respostas determinísticas, ideal para testes |
| `USE_MOCK_LLM=false` | ✅ T4 ou superior | Modelo Mistral fine-tuned via LoRA |

---

## 📋 Estrutura do Notebook
1. Clone do repositório
2. Instalação de dependências
3. Configuração de ambiente
4. Execução dos testes (~78 testes)
5. Demonstração do pipeline (4 cenários)
6. Interface Gradio (UI completa)
7. 📤 Publicar logs no Gist (para revisão do assistente)

## 1️⃣ Clone do Repositório

In [ ]:
import os

REPO_URL = 'https://github.com/felipe-huszar/medical-llm-assistant.git'
REPO_DIR = '/content/medical-llm-assistant'

if os.path.exists(REPO_DIR):
    print('📁 Repositório já existe — atualizando...')
    !cd {REPO_DIR} && git pull
else:
    print('📥 Clonando repositório...')
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print('\n✅ Repositório pronto.')
!ls -la

## 2️⃣ Instalação de Dependências

In [ ]:
!pip install -q -r requirements.txt

import importlib
print('\n📦 Verificando pacotes instalados:')
packages = [
    ('langgraph', 'LangGraph'),
    ('chromadb', 'ChromaDB'),
    ('gradio', 'Gradio'),
    ('transformers', 'Transformers'),
    ('torch', 'PyTorch'),
    ('peft', 'PEFT (LoRA)'),
]
for pkg, label in packages:
    try:
        mod = importlib.import_module(pkg)
        v = getattr(mod, '__version__', 'ok')
        print(f'  ✅ {label}: {v}')
    except ImportError:
        print(f'  ❌ {label}: NÃO instalado')

## 3️⃣ Configuração de Ambiente

In [ ]:
# Montar Drive (necessário apenas se USE_MOCK_LLM=false)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive montado.')
except Exception:
    print('ℹ️  Drive não montado (não necessário no modo Mock).')

In [ ]:
import os, sys

# ──────────────────────────────────────────────────────────
# ⬇️  MUDE AQUI para alternar entre MockLLM e modelo real
# ──────────────────────────────────────────────────────────
USE_MOCK = 'true'   # 'false' para usar Mistral LoRA (requer GPU)

os.environ['USE_MOCK_LLM']  = USE_MOCK
os.environ['MODEL_PATH']    = '/content/drive/MyDrive/medical_llm_lora'
os.environ['BASE_MODEL_ID'] = 'mistralai/Mistral-7B-Instruct-v0.1'
os.environ['CHROMA_DB_PATH']= '/content/chroma_db'

sys.path.insert(0, '/content/medical-llm-assistant')

print('=' * 55)
print('  🔧  CONFIGURAÇÃO ATUAL')
print('=' * 55)
print(f"  USE_MOCK_LLM   = {os.environ['USE_MOCK_LLM']}")
print(f"  MODEL_PATH     = {os.environ['MODEL_PATH']}")
print(f"  BASE_MODEL_ID  = {os.environ['BASE_MODEL_ID']}")
print(f"  CHROMA_DB_PATH = {os.environ['CHROMA_DB_PATH']}")
print('=' * 55)

## 4️⃣ Execução dos Testes

In [ ]:
# ── Testes Unitários (Safety Gate)
print('🧪 Testes Unitários — Safety Gate')
print('-' * 55)
!python -m pytest tests/unit -v --tb=short

In [ ]:
# ── Testes de Integração (Pipeline LangGraph)
print('🧪 Testes de Integração — Pipeline')
print('-' * 55)
!python -m pytest tests/integration -v --tb=short

In [ ]:
# ── Testes E2E (Jornadas Completas + Casos Extendidos)
print('🧪 Testes E2E — Jornadas Completas')
print('-' * 55)
!python -m pytest tests/e2e -v --tb=short

In [ ]:
# ── Sumário consolidado de todos os testes
print('📊 SUMÁRIO COMPLETO')
print('-' * 55)
!python -m pytest tests/ --tb=no -q

## 5️⃣ Demonstração do Pipeline

In [ ]:
from src.graph.pipeline import run_consultation
from src.rag.patient_rag import seed_from_file, get_consultation_history

n = seed_from_file('data/patients_seed.json')
print(f'✅ Pacientes seed carregados: {n}')
print('   (Maria Silva, João Pereira, Ana Costa)')

In [ ]:
# ── Cenário 1: Sintomas Gastrointestinais
print('🩺 CONSULTA 1 — Sintomas Gastrointestinais')
print('Paciente: Maria Silva (CPF 123.456.789-00)')
print('=' * 55)

r1 = run_consultation(
    cpf='123.456.789-00',
    doctor_question=(
        'Paciente relata dores abdominais ao evacuar há 3 semanas, '
        'com sangue nas fezes ocasionalmente. '
        'Quais diagnósticos e exames considerar?'
    ),
)
print(f"Safety: {'✅ Aprovado' if r1['safety_passed'] else '⚠️ Escalado'}  |  "
      f"Escalação: {r1['needs_escalation']}")
print()
print(r1['final_answer'])

In [ ]:
# ── Cenário 2: Retorno com Histórico
print('🩺 CONSULTA 2 — Retorno (mesmo paciente, com histórico)')
print('=' * 55)

r2 = run_consultation(
    cpf='123.456.789-00',
    doctor_question=(
        'Retorno: colonoscopia revelou pólipos adenomatosos. '
        'Dor persiste. Quais próximos passos?'
    ),
)
hist = get_consultation_history('123.456.789-00')
print(f"Safety: {'✅ Aprovado' if r2['safety_passed'] else '⚠️ Escalado'}  |  "
      f"Histórico acumulado: {len(hist)} consulta(s)")
print()
print(r2['final_answer'])

In [ ]:
# ── Cenário 3: Sintomas Cardiovasculares
print('🩺 CONSULTA 3 — Dor Torácica')
print('Paciente: João Pereira (CPF 987.654.321-00)')
print('=' * 55)

r3 = run_consultation(
    cpf='987.654.321-00',
    doctor_question=(
        'Dor torácica em aperto com irradiação para braço esquerdo, '
        'dispneia ao esforço. ECG com supra de ST em V3-V5.'
    ),
)
print(f"Safety: {'✅ Aprovado' if r3['safety_passed'] else '⚠️ Escalado'}")
print()
print(r3['final_answer'])

In [ ]:
# ── Cenário 4: Teste de Segurança — Prescrição Bloqueada
print('🛡️ CENÁRIO 4 — Safety Gate: Tentativa de Prescrição')
print('=' * 55)

from unittest.mock import MagicMock
import json

mock_llm = MagicMock()
mock_llm.invoke.return_value = json.dumps({
    'possible_diagnoses': ['Infecção bacteriana'],
    'recommended_exams': ['Hemocultura'],
    'reasoning': 'Prescrever amoxicilina 500mg 8/8h.',
    'sources': ['Protocolo'],
    'confidence': 0.85,
    'recommendation_type': 'prescription',
})

r4 = run_consultation(
    cpf='SAFETY.001',
    doctor_question='Prescreva antibiótico para infecção.',
    patient_profile={'nome': 'Safety Test', 'idade': 40, 'sexo': 'M', 'peso': 75},
    llm=mock_llm,
)
print(f"Safety: {'✅ Aprovado' if r4['safety_passed'] else '⚠️ BLOQUEADO — escalação ativada'}")
print(f"Prescrição bloqueada: {r4['needs_escalation']}")
print()
print(r4['final_answer'])

## 6️⃣ Interface Gradio (UI Completa)

Clique no link público gerado para abrir a interface.

**Abas disponíveis:**
- `👤 Paciente` — Busca/cadastro por CPF
- `🩺 Consulta` — Pergunta clínica + resposta da IA

In [ ]:
from app import demo
demo.launch(share=True)

## 7️⃣ 📤 Publicar Logs no Gist

Roda essa célula após executar os testes.  
Ela captura todos os outputs e publica num **Gist secreto** — só quem tem o link acessa.  
Cole o URL gerado no Discord para o assistente revisar.

> **Pré-requisito:** Adicione o secret `GITHUB_PAT` em **🔑 Secrets** (barra lateral esquerda)  
> com scope `gist` apenas — sem acesso a repositórios.

In [ ]:
import subprocess, urllib.request, json, datetime
from google.colab import userdata

# ── Lê o PAT do Colab Secrets (nunca aparece no código)
try:
    PAT = userdata.get('GITHUB_PAT')
except Exception:
    raise RuntimeError('❌ Secret GITHUB_PAT não encontrado. Adicione em 🔑 Secrets.')

# ── Captura output de todos os testes
print('🧪 Rodando suite completa para capturar logs...')
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/', '-v', '--tb=short', '--no-header'],
    capture_output=True, text=True, cwd='/content/medical-llm-assistant'
)
test_output = result.stdout + ('\n--- STDERR ---\n' + result.stderr if result.stderr.strip() else '')

# ── Metadata do run
timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')
status = '✅ PASSED' if result.returncode == 0 else '❌ FAILED'
header = f"""Medical LLM Assistant — Test Run
Timestamp : {timestamp}
Status    : {status}
Exit code : {result.returncode}
{'='*60}
"""
full_log = header + test_output

# ── Cria Gist secreto via API
payload = json.dumps({
    'description': f'Medical LLM — Test Run {timestamp}',
    'public': False,
    'files': {
        'test_output.txt': {'content': full_log}
    }
}).encode()

req = urllib.request.Request(
    'https://api.github.com/gists',
    data=payload,
    headers={
        'Authorization': f'token {PAT}',
        'Accept': 'application/vnd.github.v3+json',
        'Content-Type': 'application/json',
    },
    method='POST'
)

with urllib.request.urlopen(req) as r:
    gist = json.loads(r.read())

gist_url = gist['html_url']

print()
print('=' * 60)
print(f'  {status}')
print('=' * 60)
print(f'  📋 Gist secreto criado:')
print(f'  👉 {gist_url}')
print()
print('  Cole esse URL no Discord para o assistente revisar.')
print('=' * 60)